# HCCL 通信与 SpMV实验

## 7.1 实验概览

按顺序运行下面的代码，需要修改的参数已在代码中标出。


In [ ]:
%%bash
set -e
echo "CANN 目录：$ASCEND_HOME_PATH"
cmake --version
npu-smi info


<a id="s-07-02_hccl_rank_and_communicator"></a>


## 7.2 HCCL Rank 与通信域

### 本节学习目标

- 映射 rank 到 device
- 理解 rank table 和 root-info 初始化

MPI 章在主机进程之间传递数组，本章 HCCL 直接处理设备缓冲区。一进程对应一张 NPU：广播设备上的 x，Ascend C 计算局部 y，再用 HCCL 收集结果。


### 执行模型

运行脚本为每个进程设置 RANK_ID、RANK_SIZE、DEVICE_ID，一进程对应一 rank 和一设备。所有 rank 必须以一致参数创建 communicator。

### 初始化方式

提供 rank table 时调用集群信息初始化；单机无 rank table 时由 rank 0 生成 HcclRootInfo，通过共享文件传给其他进程。两种方式都要求 rank 同时参与。

### 预期现象与结果分析

rank/device 不匹配、communicator size 不一致或设备 IP 错误可能导致初始化失败或 collective 超时。

### 课后实践

为环境允许的设备数写出 rank/device 映射表，并列出需共同确认的配置。

参考答案见 `answer/07.02_answer.md`。

<a id="s-07-03_distributed_spmv_dataflow"></a>


## 7.3 分布式 SpMV 通信数据流

### 本节学习目标

- 解释 Broadcast 和 AllGather
- 理解固定块和 padding

### 数据划分

行块大小为 `ceil(rows/world_size)`。rank r 负责自己的连续行区间；最后一 rank 不足 chunk 的输出以零填充，使所有 rank 的 AllGather send count 一致。

### 通信路径

rank 0 使用 HCCL Broadcast 同步完整 x；每个 rank 计算局部 y；HCCL AllGather 收集固定大小块；Host 侧按全局 rows 截掉 padding。

MPI 的 Gatherv 只把完整 y 交给 root；本例 AllGather 让每个 rank 都收到所有分块。

例如 rows=10、world=4，chunk=3，四段分别为 `[y0,y1,y2]`、`[y3,y4,y5]`、`[y6,y7,y8]`、`[y9,0,0]`。每个 rank 收到 12 个元素，再保留前 10 个。


### 检查结果

每张卡用 Ascend C RTC 计算局部 y，再用 HCCL 收集结果。回读后与 CPU 参考结果比较，检查相对误差。

### 课后实践

以 rows=10、world=4 手工列出每个 rank 的区间、chunk 和 padding。

参考答案见 `answer/07.03_answer.md`。

<a id="s-07-04_hccl_spmv_implementation"></a>


## 7.4 HCCL 通信实验实现

### 本节学习目标

- 构建真实 HCCL backend
- 运行多 rank 实验并检查边界

### 运行方法

下面直接启动单卡程序。多卡时可使用工程的 `scripts/run.sh --npus N`，它为每张卡启动一个进程并保存各 rank 的日志。

### 编译 HCCL 程序

下面用 CMake 编译，`HCCL_SPMV_STUB=OFF` 要求使用真实的 ACL 和 HCCL。编译完成后再执行下一步的单卡实验。

In [ ]:
%%bash
set -euo pipefail
cd src/hccl_spmv
cmake -S . -B build -DCMAKE_BUILD_TYPE=Release -DHCCL_SPMV_STUB=OFF
cmake --build build -j


### 运行单卡检查

先用一个 rank 检查设备初始化、数据分区和结果校验。`--world-size 1` 指定只有一个进程，`--rank 0` 和 `--device 0` 指定它的编号与设备。多卡需要为每张卡启动一个进程，运行方法见[实验手册](EXPERIMENT_GUIDE.md)。

单卡输出中的 HCCL 时间包含调用和同步开销，用于检查流程，不表示跨卡链路性能。

有两张卡时可选做：在 `src/hccl_spmv` 目录执行 `bash scripts/run.sh --npus 2 --matrix U1 --warmup 3 --repeat 10`。更多配置见 `EXPERIMENT_GUIDE.md`。


In [ ]:
%%bash
set -euo pipefail
cd src/hccl_spmv
# 单卡实验保持 --world-size 1；改变输入时只修改 --matrix。
./build/bin/hccl_spmv --rank 0 --world-size 1 --device 0 --matrix U1 --warmup 3 --repeat 10


### 预期现象与结果分析

rank 0 应输出 backend、设备数、collective、ACL transfer、local Ascend C SpMV、distributed total 和正确性。host stub 仅用于流程检查。

### 课后实践

在源码中找到 HCCL 通信和本地 SpMV 的调用，说明各自处理什么数据。

参考答案见 `answer/07.04_answer.md`。


<a id="s-07-05_multi_npu_scaling_analysis"></a>


## 7.5 多 Rank 扩展性分析

### 本节学习目标

- 分析 collective 与端到端时间
- 避免错误性能归因

### 怎样读计时字段

`HCCL Communication` 包含 Broadcast、AllGather 和各自的同步；`Data Transfer` 是 H2D/D2H；`Local SpMV launch-to-complete` 包含本地 kernel 提交与等待完成。

`Synchronization` 是本地 SpMV 的等待子项，已经包含在 launch-to-complete 中，不要再加一次。Total 还包括 Host 组织数据等开销。

汇总先取每轮各 rank 的 MAX，再对轮次求平均；各阶段的 MAX 可能来自不同 rank，不能强行加成 total。性能统计用的额外 AllReduce 在本轮计时之外。

### 扩展性判断

rank 增加会减少局部行数，但完整 x Broadcast 和完整 y AllGather 不会同比缩小。小矩阵常被固定通信延迟主导，最优 rank 数依输入与拓扑而定。

### 分析结果

比较总时间，同时查看通信、数据搬运和本地计算各自的变化。


### 选读：旧版结果

`src/hccl_spmv/README.md` 保存旧版 CPU 局部计算加 HCCL 通信的参考数据。当前程序使用 Ascend C 做局部计算，本节分析应使用自己的运行结果。

### 课后实践

选择环境允许的 rank 集合，设计同一矩阵的扩展性记录表。

参考答案见 `answer/07.05_answer.md`。